# PolicyBot — a RAG system for evidence-based policy advice

A retrieval-augmented generation (RAG) assistant that helps policymakers engage with
research evidence. It is grounded in a curated base of five randomized controlled trials on
**Housing First** interventions for homelessness, and is deliberately designed to *surface
uncertainty and trade-offs* rather than hand down one-size-fits-all recommendations.

**Pipeline:** parse evidence into structured records → embed each study
(`text-embedding-3-small`) → retrieve the top-k most relevant studies for a query via
cosine similarity → generate a grounded, appropriately-hedged answer (`gpt-5-mini`).

> Course project — Harvard Kennedy School, DPI-681 (*Public Problem Solving with Generative AI*).
> Requires `OPENAI_API_KEY`.

## Setup

In [ ]:
# Install dependencies
!pip install openai numpy pandas pydantic --quiet

import json, time
import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
import os

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
print("Libraries imported and API client created.")

## Evidence base

Five high-quality RCTs/evaluations of Housing First, with structured metadata and abstracts.
This is the only knowledge the bot is allowed to draw on.

In [ ]:
# STEP 1: Paste your evidence table from Google Docs below.
# Select your entire table (header row + 5 data rows), copy it, and paste here.
# It will likely appear as tab-separated text — that's perfectly fine!

MY_EVIDENCE_TABLE = """
Study Name (including Author, Year)
Type of Study/Design
Country / Context
Intervention / Policy
Outcomes of interest
Headline result
Link to article
Housing First, Consumer Choice, and Harm Reduction for Homeless Individuals with a Dual Diagnosis (Tsemberis, S., Gulcur, L., Nakae, M., 2004).
RCT - interviews conducted every 6 months over the course of 24 months.
New York, USA - chronically homeless, mentally ill adults, most also experiencing substance abuse.
Housing First (immediate housing without treatment prerequisites) vs. treatment-first housing contingent on sobriety and treatment participation.
Housing stability, time spent homeless, consumer choice, substance use, substance abuse treatment utilization, psychiatric symptoms.
The experimental group obtained housing earlier, remained stably housed, and reported higher perceived choice.
https://joeornstein.github.io/pols-4641/readings/Tsemberis%20et%20al.%20-%202004%20-%20Housing%20First,%20Consumer%20Choice,%20and%20Harm%20Reduction.pdf
National Final Report: Cross-Site at Home/Chez Soi Project (Goering, P., 2014).
Multi-site RCT - mixed-methods evaluation.
5 cities across Canada (Vancouver, Winnipeg, Toronto, Montreal, Moncton). Homeless adults with mental illness.
Housing First vs. treatment as treatment as usual.
Housing stability, quality of life, service use, costs.
Housing First participants obtained housing faster, stayed housed longer, and showed better quality of life outcomes.
https://www.mentalhealthcommission.ca/wp-content/uploads/drupal/mhcc_at_home_report_national_cross-site_eng_2_0.pdf
Breaking the Homelessness-Jail Cycle with Housing First (Cunningham et. al, 2021).
RCT
Denver, CO, US - chronically homeless people with frequent criminal justice system interactions.
Housing First supportive housing program vs. treatment as usual.
Housing stability, jail days, arrests, service use.
Housing First increased housing stability, reduced jail stays and police interactions.
https://www.urban.org/sites/default/files/publication/104501/breaking-the-homelessness-jail-cycle-with-housing-first_1.pdf
A Multiple-City RCT of Housing First With Assertive Community Treatment for Homeless Canadians With Serious Mental Illness (Aubry et. al, 2016).
RCT, 950 participants.
5 Canadian cities - participants with severe mental illness who were either absolutely homeless or precariously housed.
Housing First with assertive community treatment (ACT) vs. treatment as usual.
Housing stability, quality of life, community functioning.
Housing First participants spent more time in stable housing, and experienced higher quality housing. After the first year, they experienced greater gains in community functioning and quality of life.
https://psychiatryonline.org/doi/10.1176/appi.ps.201400587?url_ver=Z39.88-2003&rfr_id=ori:rid:crossref.org&rfr_dat=cr_pub%20%200pubmed
Housing First for Homeless People with Severe Mental Illness (Loubiere et. al, 2022).
Multicentre RCT.
4 French cities - Lille, Marseille, Paris, Toulouse. Participants were homeless and diagnosed with either bipolar or schizophrenia.
Housing First with immediate access to independent housing and ACT support vs. treatment as usual.
Personal recovery, housing stability, quality of life, global physical and mental status, inpatient days, mental symptoms, addictions.
HF participants had better autonomy and housing stability, improved mental composite scores and fewer inpatient days. However, they also experienced higher alcohol consumption between baseline and 48 months.
https://pmc.ncbi.nlm.nih.gov/articles/PMC8851060/
"""


# STEP 2: For each paper, paste the title and abstract (or a hand-written summary).
# Follow this format — one entry per paper:
#
#   Title: [paper title]
#   Summary: [abstract or your brief summary]
#
#   Title: [next paper title]
#   Summary: [abstract or your brief summary]
#   ... and so on for all 5 papers

MY_SUMMARIES = """
Title: [Housing First, Consumer Choice, and Harm Reduction for Homeless Individuals with a Dual Diagnosis]
Summary: [We examined the longitudinal effects of a Housing First program for homeless, mentally ill individuals' on those individuals' consumer choice, housing stability, substance use, treatment utilization, and psychiatric symptoms. Two hundred twenty-five participants were randomly assigned to receive housing contingent on treatment and sobriety (control) or to receive immediate housing without treatment prerequisites (experimental). Interviews were conducted every 6 months for 24 months. The experimental group obtained housing earlier, remained stably housed, and reported higher perceived choice. Utilization of substance abuse treatment was significantly higher for the control group, but no differences were found in substance use or psychiatric symptoms. Participants in the Housing First program were able to obtain and maintain independent housing without compromising psychiatric or substance abuse symptoms.]

Title: [National Final Report: Cross-Site at Home/Chez Soi Project]
Summary: [This report documents the final results of the At Home/Chez Soi research demonstration project, which examined Housing First as a means of ending homelessness for people living with mental illness in Canada. The project followed more than 2,000 participants for two years, and was the world’s largest trial of Housing First, with demonstration sites in Vancouver, Winnipeg, Toronto, Montréal, and Moncton.]

Title: [Breaking the Homelessness–Jail Cycle With Housing First: Results From the Denver Supportive Housing Social Impact Bond Initiative]
Summary: [This document reports on the City and County of Denver launching of a supportive housing initiative to increase housing stability and decrease jail stays among people who experienced long-term homelessness and had frequent interactions with the criminal justice and emergency health systems.]

Title: [A Multiple-City RCT of Housing First With Assertive Community Treatment for Homeless Canadians With Serious Mental Illness]
Summary: [Housing First with assertive community treatment (ACT) is a promising approach to assist people with serious mental illness to exit homelessness. The article presents two-year findings from a multisite trial on the effectiveness of Housing First with ACT. Conclusions: Housing First with ACT is an effective approach in various contexts for assisting individuals with serious mental illness to rapidly exit homelessness.]

Title: [Housing First for Homeless People with Severe Mental Illness]
Summary: [Housing First (HF), a recovery-oriented approach, was proven effective in stabilising housing situations of homeless individuals with severe mental disorders, yet had limited effectiveness on recovery outcomes on a short-term basis compared to standard treatment. The objective was to assess the effects of the HF model among homeless people with high support needs for mental and physical health services on recovery, housing stability, quality of life, health care use, mental symptoms and addiction issues on 4 years of data from the Un Chez Soi d'Abord trial.]
"""

In [ ]:
# ── Parse your evidence — run this cell; do not edit it ───────────────────────

class Paper(BaseModel):
    """Structured representation of a single research paper or study."""
    title: str = Field(description="Title of the study or paper")
    authors: str = Field(description="Authors of the study (e.g. 'Smith et al.')")
    year: str = Field(description="Year of publication")
    study_type: str = Field(description="Type of study or research design (e.g. RCT, quasi-experimental, meta-analysis)")
    country_context: str = Field(description="Country or context where the study was conducted")
    intervention: str = Field(description="The intervention or policy studied")
    outcomes: str = Field(description="Key outcomes of interest")
    headline_result: str = Field(description="Main finding or headline result")
    summary: str = Field(description="Abstract or summary of the paper")

class PaperCollection(BaseModel):
    """A collection of parsed research papers."""
    papers: list[Paper]


def _parse_evidence(table_text: str, summaries_text: str) -> list[Paper] | None:
    """Parse the pasted table and summaries into structured Paper objects."""
    prompt = (
        "You are given two pieces of text.\n\n"
        "EVIDENCE TABLE (may be tab-separated, pipe-separated, or other table format):\n"
        f"{table_text}\n\n"
        "SUMMARIES (title/abstract or title/summary pairs):\n"
        f"{summaries_text}\n\n"
        "Instructions:\n"
        "1. Parse each row from the evidence table into a paper.\n"
        "2. Match each summary to its corresponding paper by title (titles may not match exactly — use your best judgment).\n"
        "3. If a paper has no matching summary, use the headline result as the summary.\n"
        "4. Extract all fields: title, authors, year, study_type, country_context, intervention, outcomes, headline_result, summary.\n"
        "5. For any field that is not present in the data, make a reasonable inference from context or write 'Not specified'.\n"
    )

    for attempt in range(3):
        try:
            resp = client.beta.chat.completions.parse(
                model="gpt-5-mini",
                response_format=PaperCollection,
                messages=[
                    {"role": "system", "content": "Parse research evidence into structured paper objects. Be thorough and accurate."},
                    {"role": "user", "content": prompt},
                ],
            )
            result = resp.choices[0].message.parsed.papers
            if result and len(result) > 0:
                return result
        except Exception as e:
            if attempt < 2:
                print(f"   Parsing attempt {attempt + 1} failed, retrying in {2 ** attempt} seconds...")
                time.sleep(2 ** attempt)
            else:
                print(f"❌ Could not parse your evidence after 3 attempts.")
                print(f"   Error: {e}")
    return None


# Run the parser
print("🔄 Parsing your evidence table and summaries...")
_parsed = _parse_evidence(MY_EVIDENCE_TABLE, MY_SUMMARIES)

# Convert to a list of plain dictionaries (easier to work with!)
papers = None
if _parsed is not None:
    papers = [p.model_dump() for p in _parsed]

if papers is None:
    print("\n⚠️  Could not parse your evidence. Please check that:")
    print("   1. You pasted your table from Google Docs (it should have headers and 5 rows)")
    print("   2. You filled in the title/summary pairs for each paper")
    print("   3. Your API key is working (re-run the verification cell above)")
    print("\n   Fix the issue above, then re-run this cell.")
elif len(papers) < 5:
    print(f"\n⚠️  Only found {len(papers)} papers (expected 5).")
    print("   Please check your evidence table has all 5 rows and re-run.")
else:
    print(f"\n✅ Successfully parsed {len(papers)} papers!\n")
    # Display as a DataFrame for easy review
    papers_df = pd.DataFrame(papers)
    display_df = papers_df[["title", "authors", "year", "study_type", "country_context"]].copy()
    display_df.index = range(1, len(display_df) + 1)
    display_df.index.name = "#"
    display(display_df)
    print("\n📝 Summaries loaded:")
    for i, paper in enumerate(papers, 1):
        preview = paper["summary"][:120] + "..." if len(paper["summary"]) > 120 else paper["summary"]
        print(f"   {i}. {paper['title']}")
        print(f"      {preview}\n")

## Embeddings

Convert each study's summary into a vector so we can measure semantic similarity to a query.

In [ ]:
def get_embedding(text):
    """
    Get the embedding vector for a single piece of text.

    Parameters:
    - text (str): The text to embed

    Returns:
    - list[float]: The embedding vector (a list of numbers)
    """

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[text]
    )

    return response.data[0].embedding

In [ ]:
# ── Generate embeddings for all papers ────────────────────────────────────────
# We'll embed the summary of each paper. You don't need to edit this part!

print("🔄 Generating embeddings for your papers...")
paper_embeddings = []

for i, paper in enumerate(papers):
    summary_text = paper["summary"]
    for attempt in range(3):
        try:
            embedding = get_embedding(summary_text)
            paper_embeddings.append(embedding)
            print(f"   ✅ Paper {i+1}: {paper['title'][:50]}...")
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                print(f"   ❌ Paper {i+1} failed: {e}")
                paper_embeddings.append(None)

# Check results
_failed = [emb for emb in paper_embeddings if emb is None]
if len(_failed) == 0 and len(paper_embeddings) == len(papers):
    print(f"\n✅ Generated embeddings for all {len(paper_embeddings)} papers!")
    print(f"   Each embedding has {len(paper_embeddings[0])} dimensions.")
else:
    print(f"\n❌ {len(_failed)} paper(s) failed. Check your get_embedding() function and try again.")

## Retrieval — cosine similarity + top-k

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """
    Compute the cosine similarity between two vectors.

    Parameters:
    - vec_a (list or np.array): First vector
    - vec_b (list or np.array): Second vector

    Returns:
    - float: Cosine similarity score between -1 and 1
    """
    # Convert to numpy arrays
    a = np.array(vec_a)
    b = np.array(vec_b)

    # Compute cosine similarity
    # Hint: Use np.dot() for the dot product and np.linalg.norm() for the magnitude
    similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

    return similarity

In [ ]:
def retrieve_relevant_papers(query, papers, embeddings, top_k=3):
    """
    Retrieve the most relevant papers for a given query.

    Parameters:
    - query (str): The user's policy question
    - papers (list[dict]): List of paper dictionaries
    - embeddings (list[list[float]]): Pre-computed embeddings for each paper
    - top_k (int): Number of top papers to return

    Returns:
    - list[tuple]: List of (paper_dict, similarity_score) tuples, sorted by relevance
    """

    # Step 1: Generate an embedding for the query
    # (Hint: use your get_embedding() function)
    query_embedding = get_embedding(query)

    # Step 2: Compute cosine similarity between the query embedding and
    # each paper embedding
    # (Hint: loop through the embeddings list and use your cosine_similarity()
    #  function for each one)
    similarities = []
    for emb in embeddings:
        score = cosine_similarity(query_embedding, emb)
        similarities.append(score)


    # Step 3: Pair each paper with its similarity score and sort by
    # similarity (highest first)
    # (Hint: use zip(papers, similarities) and sorted() with reverse=True)
    paired = list(zip(papers, similarities))
    ranked = sorted(paired, key=lambda x: x[1], reverse=True)


    # Step 4: Return the top_k results
    return ranked[:top_k]

## The PolicyBot

The system prompt encodes the design philosophy: calm and non-authoritative, explicit about
uncertainty, transparent about benefits/risks/trade-offs, and clear that it assists rather than
decides.

In [ ]:
# Write your PolicyBot system prompt below.
# This should reflect the tone and style you described in Part 2(b).
SYSTEM_PROMPT = """

Authoritative responses will be ineffective in this context because of their misleading nature. Therefore, the PolicyBot should be calm and receptive to changes from the policymaker interacting with it. It shouldn’t be definitive in its answers, and when the evidence presents uncertainty, the bot needs to clearly acknowledge this. It should avoid making recommendations that are one-size-fits-all, because of how contextual policymaking is. It should also make sure that it presents all of the existing benefits, risks, and trade-offs as transparently as possible. The policymaker will naturally possess more knowledge and context than the bot, so the bot should acknowledge this asymmetry. Overall, the PolicyBot should not be a decision-maker, but rather a tool to assist the policymaker using it.

"""

In [ ]:
# ── Helper function to format papers as context (provided for you) ────────────
def _format_context(relevant_papers):
    """Format retrieved papers into a readable context string for the LLM."""
    context_parts = []
    for i, (paper, score) in enumerate(relevant_papers, 1):
        context_parts.append(
            f"Paper {i}: {paper['title']}\n"
            f"Authors: {paper['authors']} ({paper['year']})\n"
            f"Study Type: {paper['study_type']}\n"
            f"Context: {paper['country_context']}\n"
            f"Intervention: {paper['intervention']}\n"
            f"Outcomes: {paper['outcomes']}\n"
            f"Key Finding: {paper['headline_result']}\n"
            f"Summary: {paper['summary']}"
        )
    return "\n\n---\n\n".join(context_parts)


def query_policybot(user_query, papers, embeddings, top_k=3):
    """
    The full RAG pipeline: retrieve relevant papers and generate a response.

    Parameters:
    - user_query (str): The policymaker's question
    - papers (list[dict]): List of paper dictionaries
    - embeddings (list[list[float]]): Pre-computed embeddings for each paper
    - top_k (int): Number of papers to retrieve

    Returns:
    - str: The PolicyBot's response
    """

    # Step 1: Retrieve the most relevant papers
    # (Use your retrieve_relevant_papers() function)
    relevant_papers = retrieve_relevant_papers(user_query, papers, embeddings, top_k=top_k)


    # Input whatever variable you define in Step 1 into this code
    context = _format_context(relevant_papers)

    # Step 2: Create the messages for the API call
    # You need two messages:
    #   1. A system message with your SYSTEM_PROMPT
    #   2. A user message that includes BOTH the evidence context AND the user's query
    #      For example:
    #      f"Based on the following research evidence:\n\n{context}\n\nQuestion: {user_query}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Based on the following research evidence:\n\n{context}\n\nQuestion: {user_query}"
        }
    ]


    # Step 3: Call the OpenAI chat API
    # Use client.chat.completions.create() with model="gpt-5-mini"
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )


    # Step 4: Extract and return the response text
    # (Hint: response.choices[0].message.content) Make sure to return the response text, not the response object
    return response.choices[0].message.content



# ── Test your PolicyBot with retry logic ──────────────────────────────────────
print("🔄 Testing your PolicyBot with a sample query...")

_test_query = f"What interventions have been studied for {papers[0]['intervention']}?"
_response = None

for attempt in range(3):
    try:
        _response = query_policybot(_test_query, papers, paper_embeddings)
        if _response and len(_response) > 0:
            print(f"✅ PolicyBot is working!\n")
            print(f"Query: {_test_query}\n")
            print(f"Response (first 500 chars):\n{_response[:500]}")
            if len(_response) > 500:
                print("...")
            break
    except Exception as e:
        if attempt < 2:
            wait_time = 2 ** attempt
            print(f"   Attempt {attempt + 1} failed, retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"❌ PolicyBot failed after 3 attempts.")
            print(f"   Error: {e}")
            print("\n   Please check your query_policybot() function and try again.")

## Testing

In [ ]:
# Write your 3 test queries below.
# Replace the example text with questions relevant to YOUR policy problem.

test_queries = [
    "Based on the current evidence, how confident should policymakers be in scaling Housing First programs to address chronic homelessness?",
    "What interventions have been shown to reduce chronic homelessness among people with severe mental illness?",
    "How might the effectiveness of Housing First programs vary across different cities?",
]

In [ ]:
# ── Run all test queries ──────────────────────────────────────────────────────
# Just run this cell — do not edit it!

print("🚀 Running your test queries through PolicyBot...")
print("=" * 70)

test_results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{'─' * 70}")
    print(f"📋 Query {i}: {query}")
    print(f"{'─' * 70}")

    response = None
    for attempt in range(3):
        try:
            response = query_policybot(query, papers, paper_embeddings)
            break
        except Exception as e:
            if attempt < 2:
                print(f"   Attempt {attempt + 1} failed, retrying...")
                time.sleep(2 ** attempt)
            else:
                print(f"   ❌ Failed after 3 attempts: {e}")

    if response:
        print(f"\n🤖 PolicyBot Response:\n")
        print(response)
        test_results.append({"query": query, "response": response})
    else:
        print("   ❌ No response received.")
        test_results.append({"query": query, "response": "ERROR: No response"})

print(f"\n{'=' * 70}")
print(f"✅ Done! {len(test_results)} queries processed.")
print("   You can reference these responses in Part 2(d) of the written assignment.")

## Summary

PolicyBot answers were noticeably more grounded than a plain LLM's: because retrieval forces
the model to cite specific studies, responses came with evidence and surfaced limitations
(e.g., the evidence base skews North American and centers chronically homeless, mentally ill
populations). The main risks are upstream — the bot is only as good as the studies fed into it,
and it lacks the local and lived-experience context a policymaker brings. The design treats it as
an assistant that makes evidence faster to engage with, not a decision-maker.